# **Importing Libraries**

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm

In [ ]:
from config import *

# **Read Data**

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)

# Check for Class Imbalance
train_df.head()

In [ ]:
# Create descriptive columns based on mappings
df_train['binary_type'] = df_train['diagnosis'].map(diagnosis_dict_binary.get)
df_train['type'] = df_train['diagnosis'].map(diagnosis_dict.get)

df_test['binary_type'] = df_test['diagnosis'].map(diagnosis_dict_binary.get)
df_test['type'] = df_test['diagnosis'].map(diagnosis_dict.get)

df_val['binary_type'] = df_val['diagnosis'].map(diagnosis_dict_binary.get)
df_val['type'] = df_val['diagnosis'].map(diagnosis_dict.get)

df_train.binary_type = df_train.binary_type.map({'DR': 1, 'No_DR': 0})
df_test.binary_type = df_test.binary_type.map({'DR': 1, 'No_DR': 0})
df_val.binary_type = df_val.binary_type.map({'DR': 1, 'No_DR': 0})

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(y='binary_type', data=df_val, order=df_val['binary_type'].value_counts().index, palette='viridis')
plt.title('Binary Classification Distribution (No_DR vs DR)', fontsize=14)
plt.xlabel('Number of Images')
plt.ylabel('Binary Type')
plt.show()

In [ ]:
# Create working directories for train/val/test

train_dir = os.path.join(NEW_DIR, 'train')
val_dir = os.path.join(NEW_DIR, 'val')
test_dir = os.path.join(NEW_DIR, 'test')

if os.path.exists(NEW_DIR):
    shutil.rmtree(NEW_DIR)

if os.path.exists(train_dir):
    shutil.rmtree(train_dir)
os.makedirs(train_dir)

if os.path.exists(val_dir):
    shutil.rmtree(val_dir)
os.makedirs(val_dir)

if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir)

# **Create new direcory & copying images into it**

In [ ]:
print("Copying Train images...")

for _, row in tqdm(df_train.iterrows(), total=len(df_train)):
    fname = row["id_code"] + ".png"
    src = os.path.join(train_src, fname)         
    dst = os.path.join(train_dir, str(row["binary_type"]))

    os.makedirs(dst, exist_ok=True)

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print("MISSING:", src)

In [ ]:
print("Copying Validation images...")

for _, row in tqdm(df_val.iterrows(), total=len(df_val)):
    fname = row["id_code"] + ".png"
    src = os.path.join(val_src, fname)
    dst = os.path.join(val_dir, str(row["binary_type"]))

    os.makedirs(dst, exist_ok=True)

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print("MISSING:", src)

In [ ]:
print("Copying Test images...")

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    fname = row["id_code"] + ".png"
    src = os.path.join(test_src, fname)
    dst = os.path.join(test_dir, str(row["binary_type"]))

    os.makedirs(dst, exist_ok=True)

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print("MISSING:", src)

In [ ]:
def count_images(path):
    counts = {}
    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len(os.listdir(cls_path))
    return counts

print("Train balance:", count_images(os.path.join(NEW_DIR, "train")))
print("Val balance:",   count_images(os.path.join(NEW_DIR, "val")))
print("Test balance:",  count_images(os.path.join(NEW_DIR, "test")))